# 05 Gold: `risk_forecast` (the ML step)

Which open milestones are most likely to finish late in the next 90 days? This notebook trains and evaluates a small forecast, then turns it into one row per open milestone in `mfg_mc_se_sa.bluerridge_gold_pm.risk_forecast`. The app's Risk forecast card, the daily brief and Genie read that table, using its `risk_band` (High / Medium / Low). A second small table, `schedule_overdue`, lists milestones that are open and already late, because a forecast of the future should not hide what is late today.

**Read this first.** The score is a **ranking signal** built from a trend forecast of past completions. It is not a calibrated probability. The backtest below measures how often the forecast's own 80 percent interval contained what actually happened; compare that number with 80 and read every score accordingly. Scores near 0 or 1 are overconfident.

## Method, in plain language

1. **Build a history.** The `schedule` table holds one row per milestone with static dates, not a daily delay history. So we build a time series from the milestones that are already finished (`actual_end` is set): for each **ISO week** and each **discipline** (mechanical, electrical, structural, piping, instrumentation) we take the average `slip_days` of milestones that finished that week. A week with no finished milestone has no observation, so we **forward-fill** it with the last known weekly average, starting from each discipline's first observed week. Forward-filling repeats old values, so it **understates real week-to-week volatility** and makes the series look smoother than it is. Every discipline has at least 20 non-empty weeks, so we keep one series per discipline (the code falls back to one overall series if one had fewer). Observed and filled week counts are printed below.
2. **Forecast.** We call the Databricks function **`ai_forecast`** for the next **13 weeks** (about 90 days after the as-of date, 2026-11-06), per discipline, with an 80 percent prediction interval. `ai_forecast` is a table-valued function that serverless notebook compute cannot call directly, so the notebook sends the SQL to the SQL warehouse through the Statement Execution API (the series is first saved as two small helper tables in `silver_engineering`: `risk_series`, the full series, and `risk_series_train`, the series without its last 8 weeks).
3. **Check it honestly.** Backtest: hold out the last **8 weeks**, forecast all 8 from the earlier weeks with the same method, and compare with what happened: MAE, bias (mean of forecast minus actual) and interval coverage. Only weeks with a real observation are scored, not forward-filled weeks, so the number of scored points is small; **structural had no completions in the holdout**, so it is not scored at all. A "repeat the last value" baseline is reported next to it.
4. **Project each open milestone.** For every milestone that is still open and whose `planned_end` falls in the 90 days after the as-of date, take its discipline's forecast for the week of its planned end as its projected slip.
5. **Upstream delay, with float (a scheduling-logic correction).** If a milestone depends on a predecessor that is still open and already late, the delay can reach it, but only the part that eats through the **float** (the slack between the predecessor's planned end and this milestone's planned start). Inherited delay = max(0, days the predecessor is late now minus float), float = successor `planned_start` minus predecessor `planned_end`. With several predecessors we take the largest inherited delay and record which predecessor drives it (`upstream_milestone_id`, `upstream_delay_days`). Example: a predecessor 15 days late with 17 days of float costs the successor nothing.
6. **Score.** `slip_probability` = P(slip > 0) from a normal distribution whose mean is the projected slip (forecast plus inherited delay) and whose spread (sigma) is backed out of the 80 percent interval width (width / (2 x 1.2816)). The inherited delay is a fixed shift and adds no uncertainty, which is another reason extreme scores are overconfident.
7. **Bands.** Fixed cutoffs chosen in advance and not adjusted afterwards: **High** if score is at least 0.75, **Medium** from 0.50 up to 0.75, **Low** below 0.50. `risk_rank` 1 is the highest score; ties go to higher criticality, then earlier planned end.
8. **Already late.** `schedule_overdue` holds open milestones whose `planned_end` is on or before the as-of date, with `days_late` and `downstream_open_count` (open milestones that depend on it directly or through other open milestones). The cryogenic tank gate MS-281 is here, not in `risk_forecast`.

In [1]:
%pip install -q mlflow
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.16.0 which is incompatible.
googleapis-common-protos 1.62.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0.dev0,>=3.19.5, but you have protobuf 7.36.2 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
import json, math, os
from datetime import date, timedelta, datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
dbutils.widgets.text("as_of_date", "2026-11-06", "As-of date")
dbutils.widgets.text("warehouse_id", "527214d7319cb124", "SQL warehouse for ai_forecast")
dbutils.widgets.text("run_id", "", "Job run id (set by the job as {{job.run_id}})")
catalog = dbutils.widgets.get("catalog"); prefix = dbutils.widgets.get("schema_prefix")
AS_OF = dbutils.widgets.get("as_of_date"); RUN_ID_PARAM = dbutils.widgets.get("run_id"); WAREHOUSE_ID = dbutils.widgets.get("warehouse_id")
SILVER = f"{catalog}.{prefix}_silver_engineering"; GOLD = f"{catalog}.{prefix}_gold_pm"
BAND_HIGH = 0.75; BAND_MEDIUM = 0.50   # fixed cutoffs, chosen before the run
HORIZON_DAYS = 90; HORIZON_WEEKS = 13; HOLDOUT_WEEKS = 8; INTERVAL = 0.8; MIN_WEEKS = 20
Z = 1.2815515655446004  # standard normal quantile for an 80 percent central interval
as_of = date.fromisoformat(AS_OF)
w = WorkspaceClient()
os.makedirs("outputs", exist_ok=True)

def wh_sql(stmt):
    r = w.statement_execution.execute_statement(statement=stmt, warehouse_id=WAREHOUSE_ID, wait_timeout="50s")
    if r.status.state.value != "SUCCEEDED":
        raise RuntimeError(f"warehouse statement {r.status.state}: {r.status.error}")
    cols = [c.name for c in r.manifest.schema.columns]
    return pd.DataFrame(r.result.data_array or [], columns=cols)
print("as_of", AS_OF, "| warehouse", WAREHOUSE_ID)

as_of 2026-11-06 | warehouse 527214d7319cb124


## 1. Weekly slip series per discipline

In [3]:
raw = spark.sql(f'''
  SELECT discipline, date_trunc('week', actual_end) AS ds, avg(slip_days) AS y, count(*) AS n_done
  FROM {SILVER}.schedule
  WHERE actual_end IS NOT NULL AND actual_end <= DATE'{AS_OF}'
  GROUP BY 1, 2''').toPandas()
raw["ds"] = pd.to_datetime(raw["ds"]); raw["y"] = raw["y"].astype(float)
last_week = pd.Timestamp(as_of) - pd.Timedelta(days=as_of.weekday())   # Monday of the as-of week
parts = []
for disc, g in raw.groupby("discipline"):
    grid = pd.date_range(g["ds"].min(), last_week, freq="7D")
    s = g.set_index("ds").reindex(grid)
    s["observed"] = s["y"].notna()
    s["y"] = s["y"].ffill()
    s["discipline"] = disc
    parts.append(s.rename_axis("ds").reset_index()[["discipline", "ds", "y", "observed"]])
series = pd.concat(parts, ignore_index=True)
summary = series.groupby("discipline").agg(weeks=("ds", "size"), observed_weeks=("observed", "sum"), first_week=("ds", "min"), last_week=("ds", "max"))
print(summary.to_string())
GROUPED = bool((summary["observed_weeks"] >= MIN_WEEKS).all())
if not GROUPED:   # fallback: one overall series
    o = raw.assign(w_=raw["y"] * raw["n_done"]).groupby("ds").agg(w_=("w_", "sum"), n=("n_done", "sum"))
    o["y"] = o["w_"] / o["n"]
    grid = pd.date_range(o.index.min(), last_week, freq="7D"); o = o.reindex(grid)
    series = pd.DataFrame({"discipline": "all", "ds": grid, "y": o["y"].ffill().values, "observed": o["y"].notna().values})
print("grouped by discipline:", GROUPED, "| total series rows:", len(series))
SERIES_LEN = int(series.groupby("discipline").size().median())

            weeks  observed_weeks first_week  last_week
discipline                                             
electrical     55              33 2025-10-20 2026-11-02
instr          55              33 2025-10-20 2026-11-02
mechanical     54              36 2025-10-27 2026-11-02
piping         54              25 2025-10-27 2026-11-02
structural     55              30 2025-10-20 2026-11-02
grouped by discipline: True | total series rows: 273


## 2. `ai_forecast` through the SQL warehouse, and the backtest

`ai_forecast` needs a SQL warehouse, so each series is saved as a table and the query is run through the Statement Execution API. The backtest uses the identical call on the series minus its last 8 weeks.

In [4]:
def save_series(df, name, comment):
    sdf = spark.createDataFrame(df[["discipline", "ds", "y"]].assign(ds=df["ds"].dt.date, y=df["y"].astype(float)))
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.{name}")
    spark.sql(f"COMMENT ON TABLE {SILVER}.{name} IS '{comment}'")

def run_ai_forecast(table, horizon_date):
    sql = (f"SELECT discipline, ds, y_forecast, y_lower, y_upper FROM ai_forecast(TABLE(SELECT discipline, ds, y FROM {SILVER}.{table}), "
           f"horizon => DATE'{horizon_date}', time_col => 'ds', value_col => 'y', group_col => 'discipline', "
           f"prediction_interval_width => {INTERVAL})")
    print("warehouse SQL:", sql)
    f = wh_sql(sql)
    f["ds"] = pd.to_datetime(f["ds"]).dt.tz_localize(None).dt.normalize()
    for c in ["y_forecast", "y_lower", "y_upper"]: f[c] = f[c].astype(float)
    return f

all_weeks = sorted(series["ds"].unique())
cut = all_weeks[-HOLDOUT_WEEKS]                       # first held-out week
train, hold = series[series["ds"] < cut], series[series["ds"] >= cut]
save_series(train, "risk_series_train", "Helper for 02_risk_forecast: pre-holdout TRAINING cut of the weekly mean slip per discipline (forward-filled), last 8 weeks removed; backtest input to ai_forecast")
save_series(series, "risk_series", "Helper for 02_risk_forecast: FULL weekly mean slip per discipline (forward-filled) up to the as-of week; input to the 13-week ai_forecast")

# horizon is exclusive: one week past the last holdout week so all 8 held-out weeks are forecast
bt = run_ai_forecast("risk_series_train", (pd.Timestamp(all_weeks[-1]) + pd.Timedelta(days=7)).date())
bt = bt.merge(hold, on=["discipline", "ds"], how="inner")
obs = bt[bt["observed"]].copy()                       # score only weeks with a real observation
obs["err"] = obs["y_forecast"] - obs["y"]
obs["inside"] = (obs["y"] >= obs["y_lower"]) & (obs["y"] <= obs["y_upper"])
# baseline: repeat the last training value of each discipline
last_train = train.sort_values("ds").groupby("discipline")["y"].last()
obs["naive_err"] = obs["discipline"].map(last_train) - obs["y"]
mae, bias, cov = float(obs["err"].abs().mean()), float(obs["err"].mean()), float(obs["inside"].mean())
naive_mae, naive_bias = float(obs["naive_err"].abs().mean()), float(obs["naive_err"].mean())
print(f"holdout weeks={HOLDOUT_WEEKS} (from {pd.Timestamp(cut).date()}), scored observed points={len(obs)} of {len(bt)}")
print(f"BACKTEST  MAE={mae:.3f} d   bias={bias:+.3f} d   coverage@{int(INTERVAL*100)}%={cov:.2%}")
print(f"BASELINE  last-value MAE={naive_mae:.3f} d   bias={naive_bias:+.3f} d")
per_disc = obs.groupby("discipline").agg(n=("err", "size"), mae=("err", lambda e: e.abs().mean()), bias=("err", "mean"), coverage=("inside", "mean")).round(3)
print(per_disc.to_string())

warehouse SQL: SELECT discipline, ds, y_forecast, y_lower, y_upper FROM ai_forecast(TABLE(SELECT discipline, ds, y FROM mfg_mc_se_sa.bluerridge_silver_engineering.risk_series_train), horizon => DATE'2026-11-09', time_col => 'ds', value_col => 'y', group_col => 'discipline', prediction_interval_width => 0.8)
holdout weeks=8 (from 2026-09-14), scored observed points=14 of 40
BACKTEST  MAE=10.624 d   bias=-9.804 d   coverage@80%=14.29%
BASELINE  last-value MAE=10.940 d   bias=-9.702 d
            n     mae    bias  coverage
discipline                             
electrical  2   4.006  -0.313     0.000
instr       4   6.826  -6.826     0.250
mechanical  6  13.874 -13.190     0.167
piping      2  15.091 -15.091     0.000


## 3. Full forecast, 13 weeks past the as-of date

In [5]:
# 90 days after 2026-11-06 is 2027-02-04, inside the week of 2027-02-01; ask for the horizon date one week beyond so that week is returned,
# then keep only weeks starting on or before as_of + 90 days (13 forecast weeks).
horizon_date = (pd.Timestamp(last_week) + pd.Timedelta(weeks=HORIZON_WEEKS + 1)).date()
fc = run_ai_forecast("risk_series", horizon_date)
fc = fc[fc["ds"] <= pd.Timestamp(as_of) + pd.Timedelta(days=HORIZON_DAYS)].reset_index(drop=True)
print("forecast rows:", len(fc), "| weeks per discipline:", fc.groupby("discipline").size().to_dict(), "| last forecast week:", fc["ds"].max().date())
print(fc.groupby("discipline")[["y_forecast", "y_lower", "y_upper"]].mean().round(2).to_string())

warehouse SQL: SELECT discipline, ds, y_forecast, y_lower, y_upper FROM ai_forecast(TABLE(SELECT discipline, ds, y FROM mfg_mc_se_sa.bluerridge_silver_engineering.risk_series), horizon => DATE'2027-02-08', time_col => 'ds', value_col => 'y', group_col => 'discipline', prediction_interval_width => 0.8)
forecast rows: 65 | weeks per discipline: {'electrical': 13, 'instr': 13, 'mechanical': 13, 'piping': 13, 'structural': 13} | last forecast week: 2027-02-01
            y_forecast  y_lower  y_upper
discipline                              
electrical        2.35    -1.33     8.50
instr             7.20    -0.37    13.34
mechanical       14.84    -0.08    29.48
piping            1.29    -1.31    14.66
structural       -0.66    -2.62     1.41


## 4. Project every open milestone

Open = no `actual_end`. In scope = `planned_end` in (as-of, as-of + 90 days]. Each gets its discipline's forecast for the week of its planned end, plus the float-aware inherited upstream delay from step 5 of the method. Planned ends on 7-8 Nov fall in the as-of week, so they use the first forecast week.

In [6]:
S = spark.sql(f'''SELECT milestone_id, name, discipline, criticality, planned_start, planned_end, actual_end, depends_on_ids
                  FROM {SILVER}.schedule''').toPandas()
for c in ("planned_start", "planned_end", "actual_end"): S[c] = pd.to_datetime(S[c])
S = S.set_index("milestone_id", drop=False)
asof_ts = pd.Timestamp(as_of)
S["is_open"] = S["actual_end"].isna()
S["days_late"] = np.where(S["is_open"] & (S["planned_end"] <= asof_ts), (asof_ts - S["planned_end"]).dt.days, 0)

def upstream(row):
    # float-aware inherited delay: max over predecessors of max(0, predecessor days late now - float)
    best, who = 0.0, None
    for p in (list(row["depends_on_ids"]) if row["depends_on_ids"] is not None else []):
        if p not in S.index or not S.loc[p, "is_open"]: continue
        late_p = float(S.loc[p, "days_late"])
        flt = (row["planned_start"] - S.loc[p, "planned_end"]).days
        inh = max(0.0, late_p - flt)
        if inh > best: best, who = inh, p
    return pd.Series({"upstream_delay_days": best, "upstream_milestone_id": who})

ms = S[S["is_open"] & (S["planned_end"] > asof_ts) & (S["planned_end"] <= asof_ts + pd.Timedelta(days=HORIZON_DAYS))].copy()
ms = pd.concat([ms, ms.apply(upstream, axis=1)], axis=1)
ms["ds"] = ms["planned_end"].apply(lambda d: d - pd.Timedelta(days=d.weekday())).clip(lower=fc["ds"].min())
ms["_g"] = ms["discipline"] if GROUPED else "all"
m = ms.merge(fc.rename(columns={"discipline": "_g"}), on=["_g", "ds"], how="left")
assert m["y_forecast"].notna().all(), "a milestone week has no forecast"
m["projected_slip_days"] = m["y_forecast"] + m["upstream_delay_days"]
m["forecast_confidence_lower"] = m["y_lower"] + m["upstream_delay_days"]
m["forecast_confidence_upper"] = m["y_upper"] + m["upstream_delay_days"]
sigma = ((m["y_upper"] - m["y_lower"]) / (2 * Z)).clip(lower=1e-6)
m["slip_probability"] = [min(1.0, max(0.0, 0.5 * (1 + math.erf(mu / (s * math.sqrt(2)))))) for mu, s in zip(m["projected_slip_days"], sigma)]
m["risk_band"] = np.where(m["slip_probability"] >= BAND_HIGH, "High", np.where(m["slip_probability"] >= BAND_MEDIUM, "Medium", "Low"))
m = m.sort_values(["slip_probability", "criticality", "planned_end"], ascending=[False, False, True]).reset_index(drop=True)
m["risk_rank"] = np.arange(1, len(m) + 1)    # strict order after the tie-break
METHOD = "ai_forecast weekly slip trend per discipline (13 wk, 80% interval) + float-aware upstream delay + normal approx P(slip>0); ranking signal, not calibrated"
out = m.assign(forecast_date=asof_ts.date(), horizon_days=HORIZON_DAYS, method=METHOD,
               computed_at=datetime.now(timezone.utc).replace(tzinfo=None), planned_end=m["planned_end"].dt.date)[
    ["milestone_id", "name", "discipline", "criticality", "planned_end", "forecast_date", "horizon_days", "projected_slip_days",
     "forecast_confidence_lower", "forecast_confidence_upper", "slip_probability", "risk_band", "risk_rank",
     "upstream_delay_days", "upstream_milestone_id", "method", "computed_at"]]
print("open milestones scored:", len(out), "| with inherited upstream delay after float:", int((m["upstream_delay_days"] > 0).sum()))
print("bands:", out["risk_band"].value_counts().reindex(["High", "Medium", "Low"]).fillna(0).astype(int).to_dict())

open milestones scored: 64 | with inherited upstream delay after float: 0
bands: {'High': 26, 'Medium': 27, 'Low': 11}


## 5. Write `risk_forecast` and `schedule_overdue` (primary key `milestone_id`)

In [7]:
spark.sql(f'''
CREATE OR REPLACE TABLE {GOLD}.risk_forecast (
  milestone_id STRING NOT NULL COMMENT 'Open milestone, primary key',
  name STRING, discipline STRING, criticality INT,
  planned_end DATE COMMENT 'Planned finish, inside (as_of, as_of + 90 days]',
  forecast_date DATE COMMENT 'The as-of date the forecast was made',
  horizon_days INT COMMENT 'Forecast horizon in days (90)',
  projected_slip_days DOUBLE COMMENT 'Projected slip in DAYS: discipline weekly-mean slip forecast for the planned-end week plus inherited upstream delay',
  forecast_confidence_lower DOUBLE COMMENT 'Lower end of the 80 percent interval, in projected slip DAYS (interval of the discipline weekly mean, shifted by upstream delay); not a bound on the probability',
  forecast_confidence_upper DOUBLE COMMENT 'Upper end of the 80 percent interval, in projected slip DAYS (interval of the discipline weekly mean, shifted by upstream delay); not a bound on the probability',
  slip_probability DOUBLE COMMENT 'RANKING SIGNAL, not a calibrated probability: P(slip > 0) by normal approximation from a trend forecast. In the backtest the 80 percent intervals covered about {cov:.0%} of held-out weeks, so values near 0 or 1 are overconfident. Display risk_band, not the raw score.',
  risk_band STRING COMMENT 'High (score >= 0.75), Medium (0.50 to <0.75) or Low (<0.50); fixed cutoffs. The field the app, brief and Genie display.',
  risk_rank INT COMMENT '1 = highest slip_probability; ties: criticality desc, planned_end asc',
  upstream_delay_days DOUBLE COMMENT 'Delay inherited from a late open predecessor after subtracting float (planned_start minus predecessor planned_end), floored at 0',
  upstream_milestone_id STRING COMMENT 'The predecessor driving upstream_delay_days, NULL when none',
  method STRING, computed_at TIMESTAMP,
  CONSTRAINT risk_forecast_pk PRIMARY KEY (milestone_id))
COMMENT 'Schedule-slip RANKING SIGNAL for open milestones due in the next 90 days: ai_forecast trend of weekly slip per discipline plus float-aware upstream delay. Not a calibrated probability: backtest 80 percent intervals covered about {cov:.0%} of held-out weeks and the backtest under-forecast slip; scores near 0 or 1 are overconfident. Use risk_band.'
''')
spark.createDataFrame(out).createOrReplaceTempView("_risk_out")
spark.sql(f'''INSERT INTO {GOLD}.risk_forecast SELECT milestone_id, name, discipline, CAST(criticality AS INT), planned_end, forecast_date, CAST(horizon_days AS INT),
  projected_slip_days, forecast_confidence_lower, forecast_confidence_upper, slip_probability, risk_band, CAST(risk_rank AS INT),
  upstream_delay_days, upstream_milestone_id, method, CAST(computed_at AS TIMESTAMP) FROM _risk_out''')
print("risk_forecast rows written:", spark.table(f"{GOLD}.risk_forecast").count())

# schedule_overdue: open milestones already past planned_end; downstream_open_count = open milestones that depend on it, directly or through other open milestones
open_ids = set(S.index[S["is_open"]])
children = {}
for mid in open_ids:
    for p in (list(S.loc[mid, "depends_on_ids"]) if S.loc[mid, "depends_on_ids"] is not None else []):
        children.setdefault(p, []).append(mid)
def downstream(mid):
    seen, stack = set(), list(children.get(mid, []))
    while stack:
        c = stack.pop()
        if c in seen: continue
        seen.add(c); stack.extend(children.get(c, []))
    return len(seen)
od = S[S["is_open"] & (S["planned_end"] <= asof_ts)].copy()
od["downstream_open_count"] = od["milestone_id"].apply(downstream)
od_out = od.assign(days_late=od["days_late"].astype(int), planned_end=od["planned_end"].dt.date, computed_at=datetime.now(timezone.utc).replace(tzinfo=None))[
    ["milestone_id", "name", "discipline", "criticality", "planned_end", "days_late", "downstream_open_count", "computed_at"]].sort_values(["days_late", "criticality"], ascending=False)
spark.sql(f'''
CREATE OR REPLACE TABLE {GOLD}.schedule_overdue (
  milestone_id STRING NOT NULL COMMENT 'Open milestone already past its planned end, primary key',
  name STRING, discipline STRING, criticality INT,
  planned_end DATE COMMENT 'Planned finish, on or before the as-of date',
  days_late INT COMMENT 'As-of date minus planned_end, in days',
  downstream_open_count INT COMMENT 'Open milestones that depend on this one, directly or through other open milestones',
  computed_at TIMESTAMP,
  CONSTRAINT schedule_overdue_pk PRIMARY KEY (milestone_id))
COMMENT 'Already late: open milestones (no actual_end) with planned_end on or before the as-of date. Companion to risk_forecast, which is forecast-only.'
''')
spark.createDataFrame(od_out).createOrReplaceTempView("_od_out")
spark.sql(f"INSERT INTO {GOLD}.schedule_overdue SELECT milestone_id, name, discipline, CAST(criticality AS INT), planned_end, CAST(days_late AS INT), CAST(downstream_open_count AS INT), CAST(computed_at AS TIMESTAMP) FROM _od_out")
display(spark.table(f"{GOLD}.schedule_overdue").orderBy(F.desc("days_late")))

risk_forecast rows written: 64


milestone_id,name,discipline,criticality,planned_end,days_late,downstream_open_count,computed_at
MS-271,Install heat exchanger - Main Substation (WP-281),mechanical,4,2026-10-22,15,60,2026-11-08T17:06:36.104Z
MS-277,Align rotating equipment - Train 3 (WP-286),mechanical,3,2026-10-26,11,19,2026-11-08T17:06:36.104Z
MS-278,Hydrotest line - Pipe Rack South (WP-282),piping,3,2026-11-01,5,42,2026-11-08T17:06:36.104Z
MS-279,Weld and inspect - Utilities Block (WP-264),piping,1,2026-11-02,4,0,2026-11-08T17:06:36.104Z
MS-281,Cryogenic Tank T-203 Install Complete,mechanical,5,2026-11-02,4,37,2026-11-08T17:06:36.104Z
MS-287,Pull instrument cable - Train 4 (WP-290),instr,1,2026-11-04,2,0,2026-11-08T17:06:36.104Z
MS-274,Align rotating equipment - Tank T-204 (WP-277),mechanical,3,2026-11-06,0,27,2026-11-08T17:06:36.104Z


## 6. Asserts

In [8]:
rf = spark.table(f"{GOLD}.risk_forecast").toPandas()
sched = S
assert len(rf) > 0
assert sched.loc[rf["milestone_id"], "actual_end"].isna().all(), "a scored milestone is not open"
assert rf["slip_probability"].between(0, 1).all()
assert ((rf["forecast_confidence_lower"] <= rf["projected_slip_days"]) & (rf["projected_slip_days"] <= rf["forecast_confidence_upper"])).all()
assert (rf["horizon_days"] == 90).all()
pe = pd.to_datetime(rf["planned_end"])
assert (pe > asof_ts).all() and (pe <= asof_ts + pd.Timedelta(days=HORIZON_DAYS)).all()
assert rf["milestone_id"].is_unique and rf["milestone_id"].notna().all()
assert sorted(rf["risk_rank"]) == list(range(1, len(rf) + 1)), "risk_rank must be dense from 1"
assert math.isfinite(mae) and math.isfinite(bias) and math.isfinite(cov)
assert (rf.sort_values("risk_rank")["slip_probability"].diff().dropna() <= 1e-12).all(), "rank must follow probability"
assert rf["risk_band"].isin(["High", "Medium", "Low"]).all()
exp_band = np.where(rf["slip_probability"] >= BAND_HIGH, "High", np.where(rf["slip_probability"] >= BAND_MEDIUM, "Medium", "Low"))
assert (rf["risk_band"].values == exp_band).all(), "risk_band inconsistent with cutoffs"
assert (rf["upstream_delay_days"] >= 0).all()
assert (rf["upstream_milestone_id"].notna() == (rf["upstream_delay_days"] > 0)).all()
ov = spark.table(f"{GOLD}.schedule_overdue").toPandas()
assert len(ov) > 0 and "MS-281" in set(ov["milestone_id"]) and (ov["days_late"] >= 0).all() and ov["milestone_id"].is_unique
assert set(ov["milestone_id"]).isdisjoint(set(rf["milestone_id"])), "overdue and forecast sets must not overlap"
print("all asserts passed:", len(rf), "open milestones in risk_forecast;", len(ov), "in schedule_overdue; score range", round(rf.slip_probability.min(), 3), "to", round(rf.slip_probability.max(), 3))

all asserts passed: 64 open milestones in risk_forecast; 7 in schedule_overdue; score range 0.313 to 0.935


## 7. Top risks and the PM summary file

In [9]:
top5 = spark.sql(f'''SELECT risk_rank, milestone_id, name, discipline, criticality, planned_end, risk_band,
  round(projected_slip_days,1) AS projected_slip_days, round(forecast_confidence_lower,1) AS lower, round(forecast_confidence_upper,1) AS upper,
  round(slip_probability,3) AS slip_probability, upstream_delay_days, upstream_milestone_id FROM {GOLD}.risk_forecast ORDER BY risk_rank LIMIT 5''')
display(top5)
band_counts = rf["risk_band"].value_counts().reindex(["High", "Medium", "Low"]).fillna(0).astype(int).to_dict()
print("band counts:", band_counts)
print("MS-281 dependents scored in risk_forecast:", rf[rf["milestone_id"].isin([i for i, d in S["depends_on_ids"].items() if d is not None and "MS-281" in list(d)])][["milestone_id", "risk_rank", "risk_band"]].to_dict("records"))

def score_txt(p): return ">95%" if p > 0.95 else f"{p:.0%}"
def row_md(r):
    up = f"+{r.upstream_delay_days:.0f} d from {r.upstream_milestone_id}" if r.upstream_delay_days > 0 else "none"
    return f"| {r.risk_rank} | {r.milestone_id} | {r['name']} | {r.discipline} | {r.criticality} | {r.planned_end} | **{r.risk_band}** ({score_txt(r.slip_probability)}) | {r.projected_slip_days:.1f} ({r.forecast_confidence_lower:.1f} to {r.forecast_confidence_upper:.1f}) | {up} |"
top = rf.sort_values("risk_rank").head(5)
ovs = ov.sort_values("days_late", ascending=False)
md_lines = [
 f"# Top 5 schedule risks, next 90 days (as of {AS_OF})", "",
 f"**How much to trust this.** The score is a ranking signal from a trend forecast, not a calibrated probability. In the 8-week backtest the forecast's 80% intervals contained the actual value for only {cov:.0%} of scored held-out weeks (target 80%), and the forecast ran {abs(bias):.1f} days {'low' if bias < 0 else 'high'} on average (MAE {mae:.1f} days over {len(obs)} scored points). Scores near 0% or 100% are overconfident, so they are shown as bands and capped at \">95%\".", "",
 f"From `{GOLD}.risk_forecast`: {len(rf)} open milestones with a planned finish between {(asof_ts + pd.Timedelta(days=1)).date()} and {(asof_ts + pd.Timedelta(days=HORIZON_DAYS)).date()}. Bands (fixed cutoffs): High = score of 0.75 or more, Medium = 0.50 to under 0.75, Low = under 0.50. Count: {band_counts['High']} High, {band_counts['Medium']} Medium, {band_counts['Low']} Low.", "",
 "| Rank | Milestone | What | Discipline | Crit. | Planned finish | Band (score) | Projected slip days (80% range) | Inherited upstream delay |",
 "|---|---|---|---|---|---|---|---|---|",
] + [row_md(r) for _, r in top.iterrows()] + ["",
 "Inherited upstream delay counts only the part of a late predecessor's delay that exceeds the float between the predecessor's planned end and this milestone's planned start.", "",
 f"## Already late (open and past planned finish, from `{GOLD}.schedule_overdue`)", "",
 "| Milestone | What | Discipline | Crit. | Planned finish | Days late | Open milestones waiting on it |", "|---|---|---|---|---|---|---|",
] + [f"| {r.milestone_id} | {r['name']} | {r.discipline} | {r.criticality} | {r.planned_end} | {r.days_late} | {r.downstream_open_count} |" for _, r in ovs.iterrows()] + ["",
 "These are not in the forecast table; they are late today.", "",
 "**Method in two sentences.** `ai_forecast` projects the weekly average slip of finished milestones for each discipline 13 weeks ahead; each open milestone takes its discipline's forecast for its planned-finish week plus any float-adjusted delay inherited from a late predecessor. The score is a normal-curve estimate of P(slip > 0) from the forecast interval, grouped into High / Medium / Low bands.", ""]
with open("outputs/risk_top5.md", "w") as fh: fh.write("\n".join(md_lines))
print("\n".join(md_lines))

risk_rank,milestone_id,name,discipline,criticality,planned_end,risk_band,projected_slip_days,lower,upper,slip_probability,upstream_delay_days,upstream_milestone_id
1,MS-335,Loop check - Train 4 (WP-315),instr,1,2027-01-25,High,8.3,0.7,14.8,0.935,0.0,
2,MS-351,Loop check - Tank T-203 (WP-364),instr,3,2027-01-21,High,8.1,0.5,14.5,0.93,0.0,
3,MS-352,Calibrate transmitters - Jetty 2 (WP-333),instr,3,2027-01-24,High,8.1,0.5,14.5,0.93,0.0,
4,MS-331,Calibrate transmitters - Utilities Block (WP-327),instr,3,2027-01-13,High,7.9,0.3,14.2,0.926,0.0,
5,MS-356,Grout equipment - Flare and Vent (WP-325),mechanical,1,2027-01-27,High,17.0,2.0,32.5,0.923,0.0,


band counts: {'High': 26, 'Medium': 27, 'Low': 11}
MS-281 dependents scored in risk_forecast: [{'milestone_id': 'MS-304', 'risk_rank': 43, 'risk_band': 'Medium'}, {'milestone_id': 'MS-340', 'risk_rank': 46, 'risk_band': 'Medium'}]
# Top 5 schedule risks, next 90 days (as of 2026-11-06)

**How much to trust this.** The score is a ranking signal from a trend forecast, not a calibrated probability. In the 8-week backtest the forecast's 80% intervals contained the actual value for only 14% of scored held-out weeks (target 80%), and the forecast ran 9.8 days low on average (MAE 10.6 days over 14 scored points). Scores near 0% or 100% are overconfident, so they are shown as bands and capped at ">95%".

From `mfg_mc_se_sa.bluerridge_gold_pm.risk_forecast`: 64 open milestones with a planned finish between 2026-11-07 and 2027-02-04. Bands (fixed cutoffs): High = score of 0.75 or more, Medium = 0.50 to under 0.75, Low = under 0.50. Count: 26 High, 27 Medium, 11 Low.

| Rank | Milestone | What | 

## 8. Chart: history, forecast with interval band, and the backtest overlay

In [10]:
discs = sorted(series["discipline"].unique())
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for ax, d in zip(axes.ravel(), discs):
    h = series[series["discipline"] == d]
    ax.plot(h["ds"], h["y"], color="#0B2026", lw=1.4, label="weekly mean slip (history)")
    ho = h[h["observed"] & (h["ds"] >= cut)]
    ax.scatter(ho["ds"], ho["y"], color="#0B2026", s=14, zorder=3)
    b = bt[bt["discipline"] == d]
    ax.fill_between(b["ds"], b["y_lower"], b["y_upper"], color="#6C8EAD", alpha=0.30, label="backtest 80% band")
    ax.plot(b["ds"], b["y_forecast"], color="#2F6F9F", lw=1.4, ls="--", label="backtest forecast")
    f = fc[fc["discipline"] == d]
    ax.fill_between(f["ds"], f["y_lower"], f["y_upper"], color="#FF3621", alpha=0.18, label="forecast 80% band")
    ax.plot(f["ds"], f["y_forecast"], color="#FF3621", lw=1.6, label="forecast (13 wk)")
    ax.axvline(asof_ts, color="#888", lw=0.8, ls=":")
    ax.set_title(d, loc="left", fontsize=11); ax.set_ylabel("slip (days)"); ax.grid(alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes.ravel()[len(discs):]: ax.axis("off")
h_, l_ = axes.ravel()[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="lower right", bbox_to_anchor=(0.97, 0.12), frameon=False)
fig.suptitle(f"Blue Ridge LNG: weekly schedule slip by discipline, backtest and 13-week ai_forecast (as of {AS_OF})", x=0.01, ha="left", fontsize=12)
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig("outputs/risk_forecast_chart.png", dpi=130); plt.close(fig)
print("saved outputs/risk_forecast_chart.png", os.path.getsize("outputs/risk_forecast_chart.png"), "bytes")

saved outputs/risk_forecast_chart.png 215112 bytes


![Weekly slip by discipline: history, backtest and 13-week ai_forecast](outputs/risk_forecast_chart.png)

The chart is committed at `05_gold/outputs/risk_forecast_chart.png`. Dark line: weekly mean slip (history). Dashed blue line and band: the 8-week backtest forecast and its 80 percent interval, with the dark dots marking the held-out weeks that had a real observation. Red line and band: the 13-week forecast from the as-of date (dotted vertical line). Where dark dots fall outside the blue band, the forecast interval missed.

## 9. MLflow and the metrics file

Everything below is written by this run. How to read it: compare the interval coverage with the 80 percent target and the model's MAE with the last-value baseline's MAE; if coverage is far below 80 or the MAE is no better than the baseline, the score is only a ranking aid and the table says so in its comments. Structural has no scored holdout points. The `job_run_id` below is the Databricks job run id passed in as a task parameter.

In [11]:
import mlflow
exp_path = "/Shared/bluerridge_pm_copilot/schedule_risk_forecast"
w.workspace.mkdirs("/Shared/bluerridge_pm_copilot")
mlflow.set_tracking_uri("databricks"); mlflow.set_registry_uri("databricks-uc")
exp = mlflow.set_experiment(exp_path)
job_run_id = RUN_ID_PARAM
if not job_run_id or "{{" in job_run_id:   # parameter not resolved: read it from the notebook context
    try: job_run_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().currentRunId().toString()
    except Exception: job_run_id = "unavailable"
print("job_run_id:", job_run_id, "| raw parameter:", repr(RUN_ID_PARAM))
params = {"method": "ai_forecast", "grain": "weekly (ISO week, Monday start)", "grouped_by": "discipline" if GROUPED else "overall",
          "series_length_weeks": SERIES_LEN, "holdout_weeks": HOLDOUT_WEEKS, "horizon_days": HORIZON_DAYS, "horizon_weeks": HORIZON_WEEKS,
          "prediction_interval_width": INTERVAL, "as_of_date": AS_OF, "invocation": "warehouse via Statement Execution API",
          "band_high": BAND_HIGH, "band_medium": BAND_MEDIUM, "upstream_adjustment": "float-aware", "job_run_id": job_run_id}
with mlflow.start_run(run_name=f"risk_forecast_{AS_OF}") as run:
    mlflow.log_params(params)
    mlflow.log_metrics({"backtest_mae_days": mae, "backtest_bias_days": bias, "backtest_interval_coverage": cov,
                        "baseline_last_value_mae_days": naive_mae, "baseline_last_value_bias_days": naive_bias,
                        "holdout_points_scored": float(len(obs)), "milestones_scored": float(len(rf)),
                        "band_high": float(band_counts["High"]), "band_medium": float(band_counts["Medium"]), "band_low": float(band_counts["Low"]),
                        "overdue_milestones": float(len(ov))})
    mlflow.log_artifact("outputs/risk_forecast_chart.png")
    mlflow_run_id = run.info.run_id
metrics = {
    "method": "ai_forecast", "method_detail": METHOD, "ai_forecast_path": "SQL warehouse via Statement Execution API (table-valued function; series saved as tables first)",
    "series_grain": "weekly (ISO week) mean slip_days of milestones finished that week, forward-filled for empty weeks",
    "grouped_by": "discipline" if GROUPED else "overall", "series_length_weeks_median_per_group": SERIES_LEN,
    "observed_weeks_per_discipline": {k: int(v) for k, v in summary["observed_weeks"].items()},
    "holdout_weeks": HOLDOUT_WEEKS, "holdout_start": str(pd.Timestamp(cut).date()), "holdout_points_scored": int(len(obs)), "holdout_points_total": int(len(bt)),
    "mae_days": round(mae, 4), "bias_days": round(bias, 4), "interval_coverage": round(cov, 4),
    "baseline_last_value_mae_days": round(naive_mae, 4), "baseline_last_value_bias_days": round(naive_bias, 4),
    "per_discipline": json.loads(per_disc.reset_index().to_json(orient="records")),
    "horizon_days": HORIZON_DAYS, "horizon_weeks": HORIZON_WEEKS, "as_of_date": AS_OF, "prediction_interval_width": INTERVAL,
    "milestones_scored": int(len(rf)), "risk_band_cutoffs": {"High": f">= {BAND_HIGH}", "Medium": f">= {BAND_MEDIUM} and < {BAND_HIGH}", "Low": f"< {BAND_MEDIUM}"},
    "risk_band_counts": band_counts, "overdue_milestones": int(len(ov)),
    "slip_probability_note": "ranking signal from a trend forecast, not calibrated; backtest interval coverage is far below the 80% target, so extreme scores are overconfident",
    "job_run_id": job_run_id, "mlflow_experiment_path": exp_path, "mlflow_experiment_id": exp.experiment_id, "mlflow_run_id": mlflow_run_id,
}
with open("outputs/risk_model_metrics.json", "w") as fh: json.dump(metrics, fh, indent=2)
print(json.dumps(metrics, indent=2))

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


job_run_id: 942883011040522 | raw parameter: '942883011040522'
{
  "method": "ai_forecast",
  "method_detail": "ai_forecast weekly slip trend per discipline (13 wk, 80% interval) + float-aware upstream delay + normal approx P(slip>0); ranking signal, not calibrated",
  "ai_forecast_path": "SQL warehouse via Statement Execution API (table-valued function; series saved as tables first)",
  "series_grain": "weekly (ISO week) mean slip_days of milestones finished that week, forward-filled for empty weeks",
  "grouped_by": "discipline",
  "series_length_weeks_median_per_group": 55,
  "observed_weeks_per_discipline": {
    "electrical": 33,
    "instr": 33,
    "mechanical": 36,
    "piping": 25,
    "structural": 30
  },
  "holdout_weeks": 8,
  "holdout_start": "2026-09-14",
  "holdout_points_scored": 14,
  "holdout_points_total": 40,
  "mae_days": 10.6243,
  "bias_days": -9.8038,
  "interval_coverage": 0.1429,
  "baseline_last_value_mae_days": 10.9405,
  "baseline_last_value_bias_days": -9